# 1단계: LLM 판정기 모델 비교 (항만 한글 행)
런타임 → 런타임 유형 변경 → **G4 / A100 / H100 중 가장 큰 GPU** 선택 후 위에서부터 순서대로 실행하세요.
결과는 드라이브 `llm_judge/results/`에 저장되고, Claude가 그 파일을 읽어 판정합니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
D = "/content/drive/MyDrive/Projects/항만문제해결을위한AI기술공모전/llm_judge"
os.chdir(D); sys.path.insert(0, D)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


In [2]:
!pip -q install transformers==4.57.6 accelerate==1.9.0 bitsandbytes==0.46.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


Gemma도 비교하려면 아래 칸에 Hugging Face 토큰을 넣으세요(google/gemma-3-12b-it 약관 동의 필요). 없으면 비워두면 Gemma만 건너뜁니다.

In [3]:
HF_TOKEN = ""   # 선택
if HF_TOKEN:
    from huggingface_hub import login; login(HF_TOKEN)
MODELS = ["Qwen/Qwen3-14B", "Qwen/Qwen3-8B", "kakaocorp/kanana-1.5-8b-instruct-2505", "Qwen/Qwen2.5-14B-Instruct"]
if HF_TOKEN:
    MODELS.append("google/gemma-3-12b-it")

In [4]:
import json, time, gc, torch, importlib
import judge_llm; importlib.reload(judge_llm)
K = 8
packets = [json.loads(l) for l in open(f"{D}/packets/eval_packets.jsonl")]
items = [(p["query"], p["cands"][:K]) for p in packets]
print(len(packets), "packets")
for name in MODELS:
    out = f"{D}/results/{name.replace('/', '__')}.jsonl"
    if os.path.exists(out):
        print("skip (done)", name); continue
    t = time.time()
    try:
        j = judge_llm.Judge(name)
    except Exception as e:
        print("LOAD FAIL", name, repr(e)[:300]); continue
    lp = j.score_many(items, batch=16)
    with open(out, "w") as f:
        for p, v in zip(packets, lp):
            f.write(json.dumps({"pid": p["pid"], "lp": [round(float(x), 5) for x in v]}) + "\n")
    print(name, f"{time.time()-t:.0f}s")
    del j; gc.collect(); torch.cuda.empty_cache()
    import shutil; shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--" + name.replace("/", "--")), ignore_errors=True)

2164 packets


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00007-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

model-00006-of-00008.safetensors:   0%|          | 0.00/4.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-14B-Instruct 286s


In [5]:
# 요약: 어려운 행(gap<4)에서 랭커 1위 vs LLM 선택 정확도 (조건별)
import numpy as np, collections, glob
for fn in sorted(glob.glob(f"{D}/results/*.jsonl")):
    res = {json.loads(l)["pid"]: json.loads(l)["lp"] for l in open(fn)}
    tab = collections.defaultdict(lambda: [0, 0, 0])
    for p in packets:
        if p["pid"] not in res: continue
        c = p["cands"][:K]; t = p["target"]
        key = (p["cond"], "hard" if p["gap"] < 4 else "easy")
        tab[key][0] += 1
        tab[key][1] += c[0]["i"] == t
        tab[key][2] += c[int(np.argmax(res[p["pid"]]))]["i"] == t
    print(os.path.basename(fn))
    for k in sorted(tab):
        n, a, b = tab[k]; print(f"   {k[0]:14s} {k[1]:4s} n={n:4d}  ranker {a/n:.3f}  LLM {b/n:.3f}  ({b-a:+d})")

Qwen__Qwen2.5-14B-Instruct.jsonl
   clean+keep     easy n= 115  ranker 1.000  LLM 0.983  (-2)
   clean+keep     hard n=  58  ranker 0.931  LLM 0.586  (-20)
   corrupt+keep   easy n=  72  ranker 0.986  LLM 0.875  (-8)
   corrupt+keep   hard n= 478  ranker 0.738  LLM 0.471  (-128)
   corrupt+s15    easy n=  58  ranker 0.966  LLM 0.707  (-15)
   corrupt+s15    hard n= 684  ranker 0.497  LLM 0.316  (-124)
   jamo2+s15      easy n=  68  ranker 1.000  LLM 0.853  (-10)
   jamo2+s15      hard n= 346  ranker 0.754  LLM 0.454  (-104)
   mix+s15        easy n=  87  ranker 1.000  LLM 0.713  (-25)
   mix+s15        hard n= 198  ranker 0.753  LLM 0.591  (-32)
Qwen__Qwen3-14B.jsonl
   clean+keep     easy n= 115  ranker 1.000  LLM 0.948  (-6)
   clean+keep     hard n=  58  ranker 0.931  LLM 0.690  (-14)
   corrupt+keep   easy n=  72  ranker 0.986  LLM 0.889  (-7)
   corrupt+keep   hard n= 478  ranker 0.738  LLM 0.527  (-101)
   corrupt+s15    easy n=  58  ranker 0.966  LLM 0.655  (-18)
   corrupt+s15 